In [ ]:
# ============================================================
# Cell 0: 프로젝트 소스코드 업로드
# 로컬에서 moe_code.zip 으로 압축한 프로젝트를 Colab에 업로드합니다.
# 실행 후 파일 선택 dialog에서 moe_code.zip 을 선택하세요.
# ============================================================
from google.colab import files
uploaded = files.upload()  # ← moe_code.zip 선택


In [ ]:
# ============================================================
# Cell 1: Colab 실행 환경 설정
# 1) PyTorch (CUDA 12.4) 및 필수 패키지 설치
# 2) A100 GPU 할당 확인
# 3) Google Drive 마운트 (korean_chat 폴더)
# 4) 프로젝트 경로(SRC) 및 저장 디렉토리 생성
# ============================================================
!pip install torch --index-url https://download.pytorch.org/whl/cu124 -q --upgrade
!pip install accelerate datasets tokenizers wandb -q

import torch
assert torch.cuda.is_available()
print(f"✅ {torch.cuda.get_device_name()} / {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import sys, os
SRC = "/content/drive/MyDrive/korean_chat"
CKPT_DIR = f"{SRC}/checkpoints"
LOG_DIR = f"{SRC}/logs"
DATA_DIR = f"{SRC}/chat_data"
TOKENIZER_DIR = f"{SRC}/tokenizer"

# 프로젝트 code/ 디렉토리를 Python path에 등록 (모듈 import용)
sys.path.insert(0, f"{SRC}/code")

for d in [CKPT_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

print("✅ Cell 1 완료")


In [ ]:
# ============================================================
# Cell 2: [선택] 경로 및 파일 존재 여부 확인
# Cell 1이 정상 실행되었는지 검증합니다.
# 필요 없으면 건너뛰어도 무방합니다.
# ============================================================
import sys, os
SRC = "/content/drive/MyDrive/korean_chat"
print("SRC exists:", os.path.exists(SRC))
print("code/ exists:", os.path.exists(f"{SRC}/code"))
print("model/ exists:", os.path.exists(f"{SRC}/code/model"))
print("dense_transformer.py:", os.path.exists(f"{SRC}/code/model/dense_transformer.py"))
print()
print("sys.path entries:")
for p in sys.path:
    if "drive" in p.lower() or "chat" in p.lower():
        print(f"  ✅ {p}")
    else:
        print(f"     {p}")


In [ ]:
# ============================================================
# Cell 3: 업로드된 소스코드 ZIP → Drive로 복사
# Cell 0에서 업로드한 moe_code.zip 을 압축 해제한 뒤,
# Google Drive의 {SRC}/code/ 에 복사합니다.
# 이미 code/ 폴더가 있으면 덮어씁니다.
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, shutil

SRC = "/content/drive/MyDrive/korean_chat"

# 업로드한 압축파일 해제 후 Drive로 복사
zip_filename = "moe_code.zip"  # ← Cell 0에서 업로드한 파일명
if os.path.exists(zip_filename):
    print(f"Extracting {zip_filename}...")
    with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
        zip_ref.extractall("/content/project_code")

    # Drive의 code 폴더로 복사 (기존 폴더 삭제 후)
    if os.path.exists(f"{SRC}/code"):
        shutil.rmtree(f"{SRC}/code")
    shutil.copytree("/content/project_code", f"{SRC}/code")
    print(f"✅ Code copied to {SRC}/code")

# 하위 디렉토리 생성
for d in ["checkpoints", "logs", "tokenizer", "chat_data", "logs/events"]:
    os.makedirs(f"{SRC}/{d}", exist_ok=True)
print(f"✅ Directories ready at {SRC}")


In [ ]:
# ============================================================
# Cell 4: AI Hub 데이터 압축 해제 + 한국어 BPE 토크나이저 학습
# 1) Google Drive에 저장된 AI Hub TL_session*.zip 파일들을
#    Colab 로컬 디스크로 압축 해제 (Drive FUSE 속도 지연 방지)
# 2) AI Hub + HuggingFace 데이터셋 샘플로 한국어 BPE 토크나이저 학습
#    (HF 데이터셋은 각 10,000행씩 샘플링하여 vocab 커버리지 확보)
# ============================================================
import os, glob, zipfile

SRC = "/content/drive/MyDrive/korean_chat"
AIHUB_ZIP_DIR = "/content/drive/MyDrive/aihub_data"     # Drive에 업로드한 AI Hub ZIP 파일 경로
EXTRACT_DIR = "/content/aihub_extracted"                # Colab 로컬 디스크 (I/O 속도 향상)

# ---- 1. AI Hub ZIP 압축 해제 (최초 1회만 실행) ----
if not os.path.exists(EXTRACT_DIR):
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    zip_files = glob.glob(os.path.join(AIHUB_ZIP_DIR, "TL_session*.zip"))
    if not zip_files:
        print(f"⚠️ {AIHUB_ZIP_DIR} 에서 TL_session*.zip 파일을 찾을 수 없습니다.")
    for zip_path in zip_files:
        print(f"Extracting {zip_path} to {EXTRACT_DIR}...")
        try:
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(EXTRACT_DIR)
        except Exception as e:
            print(f"❌ Failed: {zip_path} → {e}")
    print("✅ AI Hub extraction complete!")
else:
    print(f"✅ Already extracted: {EXTRACT_DIR}")

# ---- 2. 한국어 BPE 토크나이저 학습 ----
!python {SRC}/code/tokenizer/train_tokenizer.py \
    --korean \
    --data_dir {EXTRACT_DIR} \
    --hf_datasets \
        beomi/KoAlpaca-v1.1a \
        nlpai-lab/kullm-v2 \
        beomi/KoAlpaca-RealQA \
        kyujinpy/KOR-OpenOrca-Platypus-v3 \
        JaeJiMin/korean_chat_friendly \
        FreedomIntelligence/sharegpt-korean \
        BAEM1N/nanochat_korean \
    --hf_samples 10000 \
    --output_dir {SRC}/tokenizer


In [ ]:
# ============================================================
# Cell 5: 데이터 전처리 — AI Hub + HuggingFace 데이터셋 병합
# 모든 데이터를 <user>/<sep>/<assistant>/</s> 포맷으로 통일한 뒤,
# block_size=512 단위로 토크나이징하여 train.npy / val.npy 로 저장합니다.
#
# 사용 데이터셋 (총 7개):
#   • beomi/KoAlpaca-v1.1a        — 네이버 지식iN Q&A (instruction/output)
#   • nlpai-lab/kullm-v2           — 한국어 instruction 데이터 (152K rows)
#   • beomi/KoAlpaca-RealQA       — 실제 사용자 질문 기반 KoAlpaca
#   • kyujinpy/KOR-OpenOrca-Platypus-v3 — Orca 스타일 한국어 데이터
#   • JaeJiMin/korean_chat_friendly — 짧은 질문/답변 쌍
#   • FreedomIntelligence/sharegpt-korean — ShareGPT 대화 데이터
#   • BAEM1N/nanochat_korean      — 대규모 한국어 대화 (202K rows, ~30GB)
# ============================================================
import os

SRC = "/content/drive/MyDrive/korean_chat"
EXTRACT_DIR = "/content/aihub_extracted"

!python {SRC}/code/train/prepare_chat_data.py \
    --data_dir {EXTRACT_DIR} \
    --hf_datasets \
        beomi/KoAlpaca-v1.1a \
        nlpai-lab/kullm-v2 \
        beomi/KoAlpaca-RealQA \
        kyujinpy/KOR-OpenOrca-Platypus-v3 \
        JaeJiMin/korean_chat_friendly \
        FreedomIntelligence/sharegpt-korean \
        BAEM1N/nanochat_korean \
    --tokenizer_dir {SRC}/tokenizer \
    --output_dir {SRC}/chat_data \
    --block_size 512


In [ ]:
# ============================================================
# Cell 6: 모델 학습 (Fine-Tuning) — korean_chat_v5
#
# 전처리된 train.npy 로 Dense Transformer 162M 모델을 학습합니다.
#
# [v5 변경사항]
#   • epochs: 1 → 3     (loss 4.2에서 미수렴, 추가 학습 필요)
#   • warmup_steps: 500 → 200  (1epoch 대비 20%→8%로 최적화)
#   • dropout: 0.1 → 0.15     (3epoch 과적합 방지)
#   • run_id: korean_chat_v5
#
# [v4 학습 결과]
#   • Loss: 7.0 → 4.2 (PPL: 1,097 → 66.7)
#   • 출력 품질: 이전 대비 극적 개선 (다양한 도메인 응답 가능)
#   • 단, loss가 아직 수렴하지 않아 추가 학습 필요 확인
# ============================================================
import os
import wandb

# wandb 로그인 (최초 1회, 주석 해제 후 API 키 입력)
# wandb.login()

SRC = "/content/drive/MyDrive/korean_chat"

!PYTHONPATH={SRC}/code python -m train.train \
    --run_id korean_chat_v5 \
    --name korean_chat_model_v5 \
    --data_dir {SRC}/chat_data \
    --tokenizer_dir {SRC}/tokenizer \
    --project_dir {SRC} \
    --batch_size 32 \
    --block_size 512 \
    --lr 3e-4 \
    --epochs 3 \
    --save_every 1000 \
    --log_every 100 \
    --warmup_steps 200 \
    --dropout 0.15 \
    --wandb


In [ ]:
# ============================================================
# Cell 7: 인사말 Fine-Tuning — v5 checkpoint + greeting → v6
# [목적] "안녕" 입력 시 "질문자님의 궁금증을..." 지식iN 패턴 대신
#       자연스러운 인사말("안녕하세요!")이 나오도록 200쌍 추가 학습
# [방법] v5 checkpoint(step22000) 위에 LR 5e-5로 1epoch fine-tune
# [결과] 기존 학습 지식 유지 + 인사말 처리만 추가 학습
# ============================================================
import sys, os
SRC = "/content/drive/MyDrive/korean_chat"
sys.path.insert(0, f"{SRC}/code")

!python {SRC}/code/train/greeting_finetune.py


In [ ]:
# ============================================================
# Cell 8: 학습된 모델 추론(Inference) 테스트
# [v5 튜닝 결과]
#   - 기존 파라미터(temperature=0.8)에서는 "질문자님의 궁금증을..."
#     지식iN 패턴이 고정 출력됨
#   - temperature 1.0 / top_k 30 / rep_penalty 1.5 로 변경 시
#     "안녕하세요. 오늘 하루 잘 보내셨나요?" 등 자연스러운 인삿말 출력 확인
# ============================================================
import os

SRC = "/content/drive/MyDrive/korean_chat"

# ★ 학습 완료 후 체크포인트 파일명 확인 후 수정 ★
checkpoint_path = f"{SRC}/checkpoints/dense_korean_chat_v5.pt"

!python {SRC}/code/train/generate.py \
    --checkpoint {checkpoint_path} \
    --tokenizer_dir {SRC}/tokenizer \
    --prompt "안녕" \
    --chat \
    --temperature 1.0 \
    --top_k 30 \
    --top_p 0.95 \
    --repetition_penalty 1.5


In [ ]:
# ============================================================
# Cell 9: 로컬 배포용 체크포인트 변환
# Colab DDP BF16 체크포인트 → 로컬 추론용 단일 FP32 포맷 변환
# 변환된 파일은 {SRC}/export/korean_chat.pt 로 저장됩니다.
# 로컬 serve/app.py 에서 이 파일을 로드합니다.
# ============================================================
import os

SRC = "/content/drive/MyDrive/korean_chat"

# ★ Cell 8과 동일하게 실제 체크포인트 파일명으로 수정 ★
src_ckpt = f"{SRC}/checkpoints/dense_korean_chat_v5.pt"
dst_ckpt = f"{SRC}/export/korean_chat.pt"

!python {SRC}/code/train/export_for_local.py \
    --src {src_ckpt} \
    --dst {dst_ckpt}
